# Validation and Reporting

This notebook demonstrates:
1. Running validation against ground truth
2. Generating HTML reports with visualizations
3. Analyzing validation metrics
4. Interpreting quality scores

## Setup

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, display

# Setup
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.kantar.validation_runner import KantarValidationRunner
from src.kantar.reports import generate_validation_report

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Environment loaded")

## Find Available Validation Data

First, let's find existing synthetic data and validation results:

In [ ]:
# Find synthetic data files
synthetic_dir = project_root / 'data/synthetic/kantar'

print("Available Synthetic Datasets:")
print("=" * 60)

if synthetic_dir.exists():
    for study_dir in synthetic_dir.iterdir():
        if study_dir.is_dir():
            for market_dir in study_dir.iterdir():
                if market_dir.is_dir():
                    synthetic_files = list(market_dir.glob('synthetic_*.xlsx'))
                    validation_files = list(market_dir.glob('validation_*.json'))
                    
                    if synthetic_files:
                        print(f"\n{study_dir.name} / {market_dir.name}:")
                        print(f"  Synthetic files: {len(synthetic_files)}")
                        print(f"  Validation files: {len(validation_files)}")
                        
                        # Show most recent
                        if synthetic_files:
                            latest = max(synthetic_files, key=lambda p: p.stat().st_mtime)
                            print(f"  Latest: {latest.name}")
else:
    print("No synthetic data found. Generate some first using notebook 02.")

## Run Validation

Validate synthetic data against ground truth:

In [ ]:
# Configuration - UPDATE THESE PATHS
study_id = '61405445-01'  # Change to your study
market_code = 'US'  # Change to your market

# Find the latest synthetic file for this study/market
synthetic_path = project_root / f'data/synthetic/kantar/{study_id}/{market_code}'
synthetic_files = list(synthetic_path.glob('synthetic_*.xlsx'))

if not synthetic_files:
    print(f"⚠️  No synthetic files found in {synthetic_path}")
    print("   Generate synthetic data first using notebook 02")
else:
    synthetic_file = max(synthetic_files, key=lambda p: p.stat().st_mtime)
    print(f"Validating: {synthetic_file.name}")
    print(f"Study: {study_id} / Market: {market_code}\n")
    
    # Run validation
    runner = KantarValidationRunner()
    
    results = runner.validate_market(
        study_id=study_id,
        market_code=market_code,
        synthetic_path=synthetic_file
    )
    
    print(f"\n✓ Validation complete!")
    print(f"  Questions validated: {len(results['question_metrics'])}")

## View Validation Metrics

In [ ]:
# Display aggregate metrics
agg = results['aggregate_metrics']
success = results['success_criteria']

print("Validation Results:")
print("=" * 60)
print(f"\nDataset Info:")
print(f"  Ground Truth Respondents: {results['respondent_counts']['ground_truth']}")
print(f"  Synthetic Respondents: {results['respondent_counts']['synthetic']}")
print(f"  Questions Validated: {agg['questions_validated']}")

print(f"\nAggregate Metrics:")
print(f"  Mean KL Divergence: {agg['mean_kl_divergence']:.3f} (lower is better, target: < 0.20)")
print(f"  KS Similarity: {agg['ks_similarity']:.3f} (higher is better, target: > 0.85)")
if agg.get('mean_correlation'):
    print(f"  Mean Correlation: {agg['mean_correlation']:.3f} (higher is better, target: > 0.85)")

print(f"\nSuccess Criteria:")
for criterion, passed in success.items():
    status = '✓ PASS' if passed else '✗ FAIL'
    print(f"  {status} {criterion.replace('_', ' ').title()}")

## Generate HTML Report

Create a comprehensive HTML report with visualizations:

In [ ]:
# Find the validation JSON file
validation_files = list(synthetic_path.glob('validation_*.json'))
validation_json = max(validation_files, key=lambda p: p.stat().st_mtime)

print(f"Generating HTML report from: {validation_json.name}\n")

# Generate report
report_outputs = generate_validation_report(
    validation_json=validation_json,
    include_plots=True
)

print("\n✓ Report generated!")
for report_type, path in report_outputs.items():
    print(f"  {report_type}: {path}")
    
html_report_path = report_outputs['html']
print(f"\nOpen the report in your browser:")
print(f"  open {html_report_path}")

## Analyze Question-Level Metrics

Look at individual question performance:

In [ ]:
# Convert to DataFrame for analysis
df_metrics = pd.DataFrame(results['question_metrics'])

# Sort by KL divergence
df_sorted = df_metrics.sort_values('kl_divergence')

print("Top 10 Best Performing Questions (Lowest KL Divergence):")
print("=" * 60)
for idx, row in df_sorted.head(10).iterrows():
    print(f"  {row['column'][:50]}...")
    print(f"    KL: {row['kl_divergence']:.3f}  |  KS: {row.get('ks_statistic', 'N/A')}")
    print()

print("\nTop 5 Questions Needing Improvement (Highest KL Divergence):")
print("=" * 60)
for idx, row in df_sorted.tail(5).iterrows():
    print(f"  {row['column'][:50]}...")
    print(f"    KL: {row['kl_divergence']:.3f}  |  KS: {row.get('ks_statistic', 'N/A')}")
    print()

## Visualize Metrics Distribution

In [ ]:
# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Validation Metrics - {study_id} {market_code}', fontsize=16, fontweight='bold')

# 1. KL Divergence distribution
ax1 = axes[0, 0]
kl_values = df_metrics['kl_divergence'].dropna()
ax1.hist(kl_values, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axvline(0.20, color='red', linestyle='--', linewidth=2, label='Threshold (0.20)')
ax1.axvline(kl_values.mean(), color='orange', linestyle='--', linewidth=2, label=f'Mean ({kl_values.mean():.3f})')
ax1.set_xlabel('KL Divergence', fontweight='bold')
ax1.set_ylabel('Frequency', fontweight='bold')
ax1.set_title('KL Divergence Distribution')
ax1.legend()
ax1.grid(alpha=0.3)

# 2. KS Statistic distribution
ax2 = axes[0, 1]
ks_values = df_metrics['ks_statistic'].dropna()
if len(ks_values) > 0:
    ax2.hist(ks_values, bins=20, color='coral', edgecolor='black', alpha=0.7)
    ax2.axvline(ks_values.mean(), color='darkred', linestyle='--', linewidth=2, label=f'Mean ({ks_values.mean():.3f})')
    ax2.set_xlabel('KS Statistic', fontweight='bold')
    ax2.set_ylabel('Frequency', fontweight='bold')
    ax2.set_title('KS Statistic Distribution')
    ax2.legend()
    ax2.grid(alpha=0.3)

# 3. Top 10 Questions by KL
ax3 = axes[1, 0]
top_10 = df_sorted.head(10)
question_labels = [col.split(')')[-1][:20] + '...' if ')' in col else col[:20] + '...' for col in top_10['column']]
colors = ['green' if kl < 0.10 else 'orange' if kl < 0.20 else 'red' for kl in top_10['kl_divergence']]
ax3.barh(range(len(top_10)), top_10['kl_divergence'], color=colors, alpha=0.7)
ax3.set_yticks(range(len(top_10)))
ax3.set_yticklabels(question_labels, fontsize=8)
ax3.set_xlabel('KL Divergence', fontweight='bold')
ax3.set_title('Top 10 Best Performing Questions')
ax3.grid(axis='x', alpha=0.3)

# 4. Summary stats
ax4 = axes[1, 1]
ax4.axis('off')
summary_text = f"""
Validation Summary
{'='*40}

Questions Validated: {len(df_metrics)}

Mean KL Divergence: {agg['mean_kl_divergence']:.3f}
Median KL Divergence: {agg['median_kl_divergence']:.3f}

KS Similarity: {agg['ks_similarity']:.3f}

Success Criteria:
  KL < 0.20: {'✓ PASS' if success['kl_below_0_20'] else '✗ FAIL'}
  KS > 0.85: {'✓ PASS' if success['ks_similarity_above_0_85'] else '✗ FAIL'}
"""
ax4.text(0.1, 0.5, summary_text, fontsize=11, family='monospace', 
         verticalalignment='center')

plt.tight_layout()
plt.show()

# Save figure
fig_path = synthetic_path / 'validation_analysis.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Saved figure: {fig_path}")

## Compare Multiple Validation Runs

If you have multiple validation files, compare them:

In [ ]:
# Find all validation files
all_validation_files = sorted(synthetic_path.glob('validation_*.json'))

if len(all_validation_files) > 1:
    print(f"Found {len(all_validation_files)} validation runs:")
    print("=" * 60)
    
    comparison_data = []
    
    for val_file in all_validation_files:
        with open(val_file) as f:
            data = json.load(f)
        
        agg = data.get('aggregate_metrics', {})
        comparison_data.append({
            'file': val_file.name,
            'timestamp': data.get('timestamp', 'Unknown'),
            'n_synthetic': data.get('respondent_counts', {}).get('synthetic', 0),
            'kl_divergence': agg.get('mean_kl_divergence', None),
            'ks_similarity': agg.get('ks_similarity', None),
            'correlation': agg.get('mean_correlation', None)
        })
    
    df_comparison = pd.DataFrame(comparison_data)
    print(df_comparison.to_string(index=False))
    
    # Plot trend if available
    if len(df_comparison) > 1:
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(range(len(df_comparison)), df_comparison['kl_divergence'], 
                marker='o', linewidth=2, markersize=8, label='KL Divergence')
        ax.axhline(0.20, color='red', linestyle='--', alpha=0.5, label='Target (0.20)')
        ax.set_xlabel('Validation Run', fontweight='bold')
        ax.set_ylabel('Mean KL Divergence', fontweight='bold')
        ax.set_title('Validation Trend Over Time')
        ax.legend()
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("Only one validation run found. Generate more data to compare trends.")

## Interpreting Results

### KL Divergence (Distribution Similarity)
- **< 0.05**: Excellent match
- **0.05 - 0.10**: Good match
- **0.10 - 0.20**: Acceptable match
- **> 0.20**: Needs improvement

### KS Similarity (Statistical Similarity)
- **> 0.85**: Good (paper benchmark)
- **0.70 - 0.85**: Acceptable
- **< 0.70**: Needs improvement

### Correlation (Mean Alignment)
- **> 0.85**: Good (target for human test-retest reliability)
- **0.70 - 0.85**: Acceptable
- **< 0.70**: Needs improvement

## Next Steps

If validation shows issues:
1. Review anchor statements for questions with high KL divergence
2. Check if demographics distribution matches ground truth
3. Increase respondent count for more stable estimates
4. Try different LLM models (gpt-4o vs gpt-4o-mini)
5. Adjust temperature parameter for more/less deterministic responses